In [1]:
!pip install -q transformers peft datasets torchaudio scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 86.6 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which 

In [2]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.7 MB/s eta 0:00:00a 0:00:01


In [3]:
import random
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torchaudio
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoModel, AutoTokenizer,
                          Wav2Vec2Model, Wav2Vec2FeatureExtractor)
from peft import LoraConfig, get_peft_model
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from datasets import load_dataset
 
# ====== CẤU HÌNH ======
HF_DATASET   = "qdovan03/vietspeech-emo"
SAVE_DIR     = Path("/content")
AUDIO_MODEL  = "nguyenvulebinh/wav2vec2-base-vietnamese-250h"
TEXT_MODEL   = "vinai/phobert-base"
MAX_TEXT_LEN = 128
MAX_AUDIO_S  = 8
SR           = 16000
BATCH_SIZE   = 8
LR           = 2e-4
NUM_EPOCHS   = 10
HIDDEN_DIM   = 256
SEED         = 42
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
# ======================
 
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [4]:
ID2LABEL = {0:"Enjoyment", 1:"Sadness", 2:"Anger", 3:"Surprise", 4:"Other"}
NUM_LABELS = 5
EMO2VEC_TO_5 = {
    "happy":0, "sad":1, "fearful":1, "angry":2, "disgusted":2,
    "surprised":3, "neutral":4,
}

In [5]:
ds_full = load_dataset(HF_DATASET, split="train")
 
def add_label5(ex):
    ex["label5"] = EMO2VEC_TO_5.get(ex["emotion"], -1)
    return ex
ds_full = ds_full.map(add_label5)
ds_full = ds_full.filter(lambda ex: ex["label5"] != -1)
 
labels = ds_full["label5"]
print("Phân bố 5 lớp:", Counter(ID2LABEL[l] for l in labels))
 
idx = list(range(len(ds_full)))
tr_idx, tmp_idx = train_test_split(idx, test_size=0.2, stratify=labels, random_state=SEED)
va_idx, te_idx  = train_test_split(tmp_idx, test_size=0.5,
                    stratify=[labels[i] for i in tmp_idx], random_state=SEED)
tr_ds = ds_full.select(tr_idx)
va_ds = ds_full.select(va_idx)
te_ds = ds_full.select(te_idx)
print(f"train={len(tr_ds)} val={len(va_ds)} test={len(te_ds)}")

README.md:   0%|          | 0.00/858 [00:00<?, ?B/s]

data/train-00000-of-00006.parquet:   0%|          | 0.00/439M [00:00<?, ?B/s]

data/train-00001-of-00006.parquet:   0%|          | 0.00/434M [00:00<?, ?B/s]

data/train-00002-of-00006.parquet:   0%|          | 0.00/436M [00:00<?, ?B/s]

data/train-00003-of-00006.parquet:   0%|          | 0.00/439M [00:00<?, ?B/s]

...

data/train-00005-of-00006.parquet:   0%|          | 0.00/437M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/20000 [00:00<?, ? examples/s]

Phân bố 5 lớp: Counter({'Other': 9477, 'Enjoyment': 7385, 'Surprise': 1660, 'Anger': 954, 'Sadness': 524})
train=16000 val=2000 test=2000


In [6]:
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL, use_fast=False)
feat_ext  = Wav2Vec2FeatureExtractor.from_pretrained(AUDIO_MODEL)

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [7]:
class FusionDataset(Dataset):
    def __init__(self, hf_ds): self.ds = hf_ds
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        ex  = self.ds[i]
        arr = np.asarray(ex["audio"]["array"], dtype=np.float32)
        sr  = ex["audio"]["sampling_rate"]
        if sr != SR:
            arr = torchaudio.functional.resample(torch.tensor(arr), sr, SR).numpy()
        arr = arr[: MAX_AUDIO_S * SR]
        return {"wav": arr, "text": ex["transcription"], "label": ex["label5"]}
 
def collate(batch):
    wavs   = [b["wav"] for b in batch]
    texts  = [b["text"] for b in batch]
    labels = torch.tensor([b["label"] for b in batch])
    audio_in = feat_ext(wavs, sampling_rate=SR, return_tensors="pt", padding=True)
    text_in  = tokenizer(texts, padding=True, truncation=True,
                         max_length=MAX_TEXT_LEN, return_tensors="pt")
    return audio_in, text_in, labels
 
train_dl = DataLoader(FusionDataset(tr_ds), batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate)
val_dl   = DataLoader(FusionDataset(va_ds), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
test_dl  = DataLoader(FusionDataset(te_ds), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

In [8]:
class MultimodalSER(nn.Module):
    def __init__(self, num_labels, hidden=HIDDEN_DIM):
        super().__init__()
        self.audio = Wav2Vec2Model.from_pretrained(AUDIO_MODEL)
        self.audio = get_peft_model(self.audio, LoraConfig(
            r=16, lora_alpha=32, lora_dropout=0.1,
            target_modules=["q_proj", "v_proj"]))
        adim = self.audio.config.hidden_size
        self.text = AutoModel.from_pretrained(TEXT_MODEL)
        self.text = get_peft_model(self.text, LoraConfig(
            r=16, lora_alpha=32, lora_dropout=0.1,
            target_modules=["query", "value"]))
        tdim = self.text.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(adim + tdim, hidden),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, num_labels))
 
    def forward(self, audio_in, text_in):
        a = self.audio(**audio_in).last_hidden_state.mean(dim=1)
        t = self.text(**text_in).last_hidden_state[:, 0]
        return self.classifier(torch.cat([a, t], dim=-1))
 
model = MultimodalSER(NUM_LABELS).to(DEVICE)

if torch.cuda.device_count() > 1:
    print(f"Dùng {torch.cuda.device_count()} GPU")
    model = nn.DataParallel(model)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Wav2Vec2Model LOAD REPORT from: nguyenvulebinh/wav2vec2-base-vietnamese-250h
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 
lm_head.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Wav2Vec2Model does not expose input embeddings. Gradients cannot flow back to the token embeddings when using adapters or gradient checkpointing. Override `get_input_embeddings` to fully support those features, or set `_input_embed_layer` to the attribute name that holds the embeddings.


pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.bias                    | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dùng 2 GPU


In [9]:
cnt = Counter(tr_ds["label5"])
weights = torch.tensor(
    [len(tr_ds)/(NUM_LABELS*cnt[i]) for i in range(NUM_LABELS)],
    dtype=torch.float, device=DEVICE)
print("class weights:", {ID2LABEL[i]: round(weights[i].item(),2) for i in range(NUM_LABELS)})
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
 
# ---------- Cell 9: Train loop ----------
def run_epoch(dl, train=True):
    model.train() if train else model.eval()
    pred, true, tot = [], [], 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for audio_in, text_in, labels in dl:
            audio_in = {k: v.to(DEVICE) for k, v in audio_in.items()}
            text_in  = {k: v.to(DEVICE) for k, v in text_in.items()}
            labels   = labels.to(DEVICE)
            logits = model(audio_in, text_in)
            loss = criterion(logits, labels)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            tot += loss.item()
            pred += logits.argmax(-1).cpu().tolist()
            true += labels.cpu().tolist()
    return tot/len(dl), accuracy_score(true,pred), f1_score(true,pred,average="macro")
 
best = 0
for ep in range(10, 15):   # epoch 11-15
    _, tra, trf = run_epoch(train_dl, True)
    _, vaa, vaf = run_epoch(val_dl, False)
    print(f"Epoch {ep+1}: train_f1={trf:.3f} | val_acc={vaa:.3f} val_f1={vaf:.3f}")
    if vaf > best:
        best = vaf
        to_save = model.module if hasattr(model,'module') else model
        torch.save(to_save.state_dict(), SAVE_DIR / "best_fusion.pt")
        print("  -> lưu best")

class weights: {'Enjoyment': 0.54, 'Sadness': 7.64, 'Anger': 4.19, 'Surprise': 2.41, 'Other': 0.42}
Epoch 1: train_f1=0.313 | val_acc=0.379 val_f1=0.341
  -> lưu best
Epoch 2: train_f1=0.402 | val_acc=0.580 val_f1=0.487
  -> lưu best
Epoch 3: train_f1=0.460 | val_acc=0.628 val_f1=0.514
  -> lưu best
Epoch 4: train_f1=0.504 | val_acc=0.584 val_f1=0.469
Epoch 5: train_f1=0.536 | val_acc=0.664 val_f1=0.570
  -> lưu best
Epoch 6: train_f1=0.549 | val_acc=0.696 val_f1=0.567
Epoch 7: train_f1=0.565 | val_acc=0.686 val_f1=0.566
Epoch 8: train_f1=0.571 | val_acc=0.689 val_f1=0.573
  -> lưu best
Epoch 9: train_f1=0.588 | val_acc=0.686 val_f1=0.542
Epoch 10: train_f1=0.593 | val_acc=0.722 val_f1=0.588
  -> lưu best


In [11]:
state = torch.load(SAVE_DIR / "best_fusion.pt")

# nếu model đang bọc DataParallel -> load vào model.module
target = model.module if hasattr(model, "module") else model
target.load_state_dict(state)
_, ta, tf = run_epoch(test_dl, False)
print(f"\n===== FUSION TEST (audio): acc={ta:.3f} | macro-F1={tf:.3f} =====")
 
model.eval(); P, T = [], []
with torch.no_grad():
    for audio_in, text_in, labels in test_dl:
        audio_in = {k: v.to(DEVICE) for k, v in audio_in.items()}
        text_in  = {k: v.to(DEVICE) for k, v in text_in.items()}
        P += model(audio_in, text_in).argmax(-1).cpu().tolist()
        T += labels.tolist()
print(classification_report(T, P, target_names=[ID2LABEL[i] for i in range(NUM_LABELS)]))


===== FUSION TEST (audio): acc=0.743 | macro-F1=0.613 =====
              precision    recall  f1-score   support

   Enjoyment       0.77      0.74      0.75       739
     Sadness       0.49      0.42      0.45        52
       Anger       0.68      0.42      0.52        96
    Surprise       0.50      0.58      0.54       166
       Other       0.79      0.82      0.81       947

    accuracy                           0.74      2000
   macro avg       0.64      0.60      0.61      2000
weighted avg       0.74      0.74      0.74      2000



In [12]:
import os
# kiểm tra vài chỗ hay lưu
for p in ["/kaggle/working/best_fusion.pt", "best_fusion.pt", "/content/best_fusion.pt"]:
    if os.path.exists(p):
        print("Tìm thấy:", p, "-", os.path.getsize(p)/1e6, "MB")

Tìm thấy: /content/best_fusion.pt - 924.045395 MB


In [13]:
from huggingface_hub import login, HfApi
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN)

api = HfApi()
REPO = "qdovan03/vietspeech-fusion-ser"   # <-- đổi tên repo nếu muốn
api.create_repo(REPO, repo_type="model", exist_ok=True)

api.upload_file(
    path_or_fileobj="/content/best_fusion.pt",   # đúng path bạn vừa tìm thấy
    path_in_repo="best_fusion.pt",
    repo_id=REPO,
    repo_type="model",
)
print(f"Đã đẩy lên: https://huggingface.co/{REPO}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Đã đẩy lên: https://huggingface.co/qdovan03/vietspeech-fusion-ser


In [14]:
import json
config = {
    "audio_model": "nguyenvulebinh/wav2vec2-base-vietnamese-250h",
    "text_model": "vinai/phobert-base",
    "num_labels": 5, "hidden_dim": 256,
    "id2label": {0:"Enjoyment",1:"Sadness",2:"Anger",3:"Surprise",4:"Other"},
    "max_audio_s": 8, "max_text_len": 128,
}
with open("/content/fusion_config.json","w") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
api.upload_file(path_or_fileobj="/content/fusion_config.json",
                path_in_repo="fusion_config.json",
                repo_id=REPO, repo_type="model")
print("Đã đẩy config")

Đã đẩy config
